<a href="https://colab.research.google.com/github/sadiq937/Shaik_DM/blob/main/PS5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PS4 - DATAMANGEMENT

# Macro-Financial Analysis: Understanding the Relationship Between Economic Indicators and Market Trends


# Abstract

This project explores the dynamic relationship between macroeconomic indicators and stock market performance, using data sourced from the Federal Reserve Economic Data (FRED) API and Alpha Vantage API. The goal is to uncover insights into how key macroeconomic factors such as the S&P 500 Index, Consumer Price Index (CPI), Federal Funds Rate, Real GDP Growth Rate, and Unemployment Rate influence the performance of major stocks. The project combines data analysis, visualization, and fuzzy matching techniques to provide a robust understanding of the interaction between these economic variables and market trends.

Data is fetched from the FRED API for the primary macroeconomic indicators, while the stock prices of key companies are retrieved using Alpha Vantage's API. The data is then cleaned and merged to form a comprehensive dataset spanning several decades. This dataset undergoes several transformations, including the calculation of percentage changes to assess the volatility and trends of the variables over time. The data is also reshaped using techniques such as stacking and unstacking to create long-format data, allowing for a more detailed examination of the relationships between the indicators.

Fuzzy matching is employed using the fuzzywuzzy package to standardize indicator names and ensure consistency across different data sources. This technique matches slightly mismatched names, such as "CPI" to "Consumer Price Index," ensuring the data is accurately represented and improving the reliability of subsequent analysis.

For visualization, the project employs the powerful Plotly library to generate a variety of interactive and insightful graphs, including:

- **Correlation Heatmap**: This visualization illustrates the relationships between the macroeconomic indicators. Using a color-coded matrix, it highlights areas of strong or weak correlation, helping to identify how different economic factors interact with one another.
  
- **Time-Series Line Plot**: This graph depicts the trend of the S&P 500 Index over time, offering a visual representation of how it fluctuates in relation to the other macroeconomic indicators. The plot allows for a comparison of the performance of the stock market relative to key economic variables.
  
- **Scatter Plot**: By analyzing the relationship between inflation (CPI) and interest rates (Federal Funds Rate), this plot provides insights into the correlation between these two crucial economic factors. A linear trendline is overlaid to emphasize the direction and strength of the relationship.
  
- **Bar Chart**: This chart categorizes market trends based on the calculated percentage change of the S&P 500. The trends are classified as "Boom," "Downturn," or "Stable," and the bar chart visualizes the frequency of each trend, offering a clear view of market behavior over time.
  
- **Area Plot**: An area plot is used to show the overall macroeconomic trends of multiple indicators over time, offering a broad view of economic performance and illustrating how these factors collectively impact the stock market.

The project further enhances the analysis by categorizing market trends using the computed percentage change of the S&P 500, which is classified into three categories: Boom, Downturn, and Stable. This categorization helps in understanding how the stock market reacts to various economic conditions and provides insights into future market behaviors.

The findings of this project lay the groundwork for the next assignment (PS5), where the focus will be to gain deeper insights into how macroeconomic factors, particularly the CPI and Federal Funds Rate, influence stock market performance. This will be achieved through advanced statistical techniques and time-series forecasting methods, allowing for more precise predictions and analyses.

Overall, this project not only provides valuable visualizations and statistical insights into the relationships between macroeconomic indicators and stock market performance but also sets the stage for a more in-depth investigation of stock price predictions in future assignments.


## INSTALLING DEPENDENCIES

In [3]:
!pip install fuzzywuzzy

In [4]:
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.9 MB/s eta 0:00:00


In [5]:
!pip install yfinance

In [6]:
import requests
import os
import pandas as pd
import re
import time
import plotly.express as px
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import folium
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from io import StringIO


## LOADING DATASETS - API

In [7]:
# FRED API Key
FRED_API_KEY = "724bdeda0e13567e197e4ec09e79e655"

# Function to fetch data from FRED
def fetch_fred_data(series_id, start_date="2000-01-01", end_date=str(datetime.today().date())):
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json&observation_start={start_date}&observation_end={end_date}"
    response = requests.get(url).json()
    data = response.get("observations", [])
    return pd.DataFrame([{"date": obs["date"], series_id: float(obs["value"])} for obs in data if obs["value"] != "."])

In [8]:
# Alpha Vantage API Key
ALPHA_VANTAGE_API_KEY = "VFIZUTBYB83CVEXI"

In [9]:
# Function to fetch data from Alpha Vantage
def fetch_alpha_vantage_data(function, symbol, interval="Monthly", start_date="2000-01-01", end_date=str(datetime.today().date())):
    url = f"https://www.alphavantage.co/query?function={function}&symbol={symbol}&interval={interval}&apikey={ALPHA_VANTAGE_API_KEY}&datatype=csv"
    response = requests.get(url).text
    df = pd.read_csv(StringIO(response))
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df[(df['timestamp'] >= start_date) & (df['timestamp'] <= end_date)]
    return df[['timestamp', 'close']].rename(columns={'timestamp': 'date', 'close': symbol})

In [10]:
# S&P 500 Index
sp500_df = fetch_fred_data("SP500")
sp500_df.head()

,date,SP500
0,2015-04-27,2108.92
1,2015-04-28,2114.76
2,2015-04-29,2106.85
3,2015-04-30,2085.51
4,2015-05-01,2108.29


In [11]:
# Consumer Price Index
inflation_df = fetch_fred_data("CPIAUCSL")
inflation_df.head()

,date,CPIAUCSL
0,2000-01-01,169.3
1,2000-02-01,170.0
2,2000-03-01,171.0
3,2000-04-01,170.9
4,2000-05-01,171.2


In [12]:
# Fedral Funds Rate
interest_rate_df = fetch_fred_data("FEDFUNDS")
interest_rate_df.head()

,date,FEDFUNDS
0,2000-01-01,5.45
1,2000-02-01,5.73
2,2000-03-01,5.85
3,2000-04-01,6.02
4,2000-05-01,6.27


In [13]:
# Real GDP Growth Rate
gdp_df = fetch_fred_data("A191RL1Q225SBEA")
gdp_df.head()

,date,A191RL1Q225SBEA
0,2000-01-01,1.5
1,2000-04-01,7.5
2,2000-07-01,0.4
3,2000-10-01,2.4
4,2001-01-01,-1.3


In [14]:
# Unemployment Rate
unemployment_df = fetch_fred_data("UNRATE")
unemployment_df.head()

,date,UNRATE
0,2000-01-01,4.0
1,2000-02-01,4.1
2,2000-03-01,4.0
3,2000-04-01,3.8
4,2000-05-01,4.0


In [15]:
# Fetching stock data from Alpha Vantage API (APPLE)
apple_stock_df = fetch_alpha_vantage_data("TIME_SERIES_MONTHLY", "AAPL")
apple_stock_df.head()

,date,AAPL
0,2025-04-28,210.14
1,2025-03-31,222.13
2,2025-02-28,241.84
3,2025-01-31,236.00
4,2024-12-31,250.42


In [16]:
# Fetching stock data from Alpha Vantage API (NVIDIA)
nvidia_stock_df = fetch_alpha_vantage_data("TIME_SERIES_MONTHLY", "NVDA")
nvidia_stock_df.head()

,date,NVDA
0,2025-04-28,108.73
1,2025-03-31,108.38
2,2025-02-28,124.92
3,2025-01-31,120.07
4,2024-12-31,134.29


## MERGING DATASETS

In [17]:
macro_df = sp500_df.merge(inflation_df, on="date", how="inner")\
                      .merge(interest_rate_df, on="date", how="inner")\
                      .merge(gdp_df, on="date", how="inner")\
                      .merge(unemployment_df, on="date", how="inner")

macro_df["date"] = pd.to_datetime(macro_df["date"])
macro_df.set_index("date", inplace=True)
macro_df.head()

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE
date,,,,,
2015-07-01,2077.42,238.034,0.13,1.6,5.2
2015-10-01,1923.82,237.733,0.12,0.7,5.0
2016-04-01,2072.78,238.992,0.37,1.3,5.1
2016-07-01,2102.95,240.101,0.39,2.9,4.8
2018-10-01,2924.59,252.772,2.19,0.6,3.8


In [18]:
# Compute percentage changes
pct_change_df = macro_df.pct_change().dropna()
pct_change_df.head()

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE
date,,,,,
2015-10-01,-0.073938,-0.001265,-0.076923,-0.562500,-0.038462
2016-04-01,0.077429,0.005296,2.083333,0.857143,0.020000
2016-07-01,0.014555,0.004640,0.054054,1.230769,-0.058824
2018-10-01,0.390708,0.052774,4.615385,-0.793103,-0.208333
2019-04-01,-0.019627,0.009736,0.105023,4.666667,-0.026316


## STACK/ UNSTACK

In [19]:
long_format_df = pct_change_df.stack().reset_index()
long_format_df.columns = ["date", "indicator", "value"]
long_format_df.head()

,date,indicator,value
0,2015-10-01,SP500,-0.073938
1,2015-10-01,CPIAUCSL,-0.001265
2,2015-10-01,FEDFUNDS,-0.076923
3,2015-10-01,A191RL1Q225SBEA,-0.562500
4,2015-10-01,UNRATE,-0.038462


## MARKET TREND CATEGORY

In [20]:
def categorize_market(value):
    if value > 0.02:
        return "Boom"
    elif value < -0.02:
        return "Downturn"
    else:
        return "Stable"

pct_change_df["Market_Trend"] = pct_change_df["SP500"].apply(categorize_market)
pct_change_df.head()

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE,Market_Trend
date,,,,,,
2015-10-01,-0.073938,-0.001265,-0.076923,-0.562500,-0.038462,Downturn
2016-04-01,0.077429,0.005296,2.083333,0.857143,0.020000,Boom
2016-07-01,0.014555,0.004640,0.054054,1.230769,-0.058824,Stable
2018-10-01,0.390708,0.052774,4.615385,-0.793103,-0.208333,Boom
2019-04-01,-0.019627,0.009736,0.105023,4.666667,-0.026316,Stable


## RESHAPE (PIVOT)

In [21]:
pivot_df = long_format_df.pivot(index="date", columns="indicator", values="value")
pivot_df.head()

indicator,A191RL1Q225SBEA,CPIAUCSL,FEDFUNDS,SP500,UNRATE
date,,,,,
2015-10-01,-0.562500,-0.001265,-0.076923,-0.073938,-0.038462
2016-04-01,0.857143,0.005296,2.083333,0.077429,0.020000
2016-07-01,1.230769,0.004640,0.054054,0.014555,-0.058824
2018-10-01,-0.793103,0.052774,4.615385,0.390708,-0.208333
2019-04-01,4.666667,0.009736,0.105023,-0.019627,-0.026316


## CORRELATION MATRIX

In [22]:
numerical_df = pct_change_df.select_dtypes(include=['number'])
correlation_matrix_df = numerical_df.corr()
correlation_matrix_df

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE
SP500,1.000000,0.679210,0.368073,-0.035030,-0.396294
CPIAUCSL,0.679210,1.000000,0.720566,0.179440,-0.275009
FEDFUNDS,0.368073,0.720566,1.000000,0.379848,-0.291805
A191RL1Q225SBEA,-0.035030,0.179440,0.379848,1.000000,-0.723314
UNRATE,-0.396294,-0.275009,-0.291805,-0.723314,1.000000


## VISUALIZATIONS

In [23]:
fig1 = px.imshow(correlation_matrix_df, text_auto=True, color_continuous_scale='RdBu_r', title="Correlation Between Macroeconomic Indicators")
fig1.show()


## TREND OF S&P 500 over time

In [24]:
df_reset = pct_change_df.reset_index()
fig2 = px.line(df_reset, x="date", y="SP500", title="S&P 500 Trend Over Time", labels={"SP500": "S&P 500 Returns"})
fig2.show()

## Inflation vs interest rate

In [25]:

# Scatter plot of Inflation vs. Interest Rates
fig3 = px.scatter(df_reset, x="CPIAUCSL", y="FEDFUNDS", title="Inflation vs. Interest Rates", labels={"CPIAUCSL": "Inflation (CPI)", "FEDFUNDS": "Interest Rate"}, trendline="ols")
fig3.show()


## MARKET TREND BAR CHART

In [26]:
# Bar chart for market trend categories
market_trend_counts_df = pct_change_df["Market_Trend"].value_counts().reset_index()
market_trend_counts_df.columns = ["Trend", "Count"]
fig4 = px.bar(market_trend_counts_df, x="Trend", y="Count", title="Market Trend Distribution", color="Trend")
fig4.show()

## MACROECONOMIC INDICATORS OVER TIME

In [27]:
# Time-series area plot
fig5 = px.area(df_reset, x="date", y=["SP500", "CPIAUCSL", "FEDFUNDS", "A191RL1Q225SBEA", "UNRATE"], title="Macroeconomic Indicators Over Time", labels={"value": "Percentage Change", "variable": "Indicator"})
fig5.show()

## FUZZYWUZZY

In [28]:
fuzzy_data_df = pd.DataFrame({"Indicator_Names": ["CPIAUCSL", "FED Funds Rate", "SP 500", "CPI", "AAPL Stock"]})

def match_indicator_name(indicator_name):
    threshold = 50

    # Check similarity ratios for each comparison
    similarity_score_cpi = fuzz.ratio(indicator_name, "Consumer Price Index")
    similarity_score_fed = fuzz.ratio(indicator_name, "Federal Funds Rate")
    similarity_score_sp500 = fuzz.ratio(indicator_name, "S&P 500 Index")
    similarity_score_apple = fuzz.ratio(indicator_name, "Apple Stock")

    # Print similarity scores for each match attempt
    print(f"Matching '{indicator_name}' with 'Consumer Price Index' gives score: {similarity_score_cpi}")
    print(f"Matching '{indicator_name}' with 'Federal Funds Rate' gives score: {similarity_score_fed}")
    print(f"Matching '{indicator_name}' with 'S&P 500 Index' gives score: {similarity_score_sp500}")
    print(f"Matching '{indicator_name}' with 'Apple Stock' gives score: {similarity_score_apple}")

    # Determine the best match based on the highest similarity score
    if similarity_score_cpi > threshold:
        return "Consumer Price Index"
    elif similarity_score_fed > threshold:
        return "Federal Funds Rate"
    elif similarity_score_sp500 > threshold:
        return "S&P 500 Index"
    elif similarity_score_apple > threshold:
        return "Apple Stock"
    else:
        return indicator_name

fuzzy_data_df["Matched_Names"] = fuzzy_data_df["Indicator_Names"].apply(match_indicator_name)
fuzzy_data_df

Matching 'CPIAUCSL' with 'Consumer Price Index' gives score: 21
Matching 'CPIAUCSL' with 'Federal Funds Rate' gives score: 0
Matching 'CPIAUCSL' with 'S&P 500 Index' gives score: 19
Matching 'CPIAUCSL' with 'Apple Stock' gives score: 21
Matching 'FED Funds Rate' with 'Consumer Price Index' gives score: 24
Matching 'FED Funds Rate' with 'Federal Funds Rate' gives score: 75
Matching 'FED Funds Rate' with 'S&P 500 Index' gives score: 30
Matching 'FED Funds Rate' with 'Apple Stock' gives score: 16
Matching 'SP 500' with 'Consumer Price Index' gives score: 15
Matching 'SP 500' with 'Federal Funds Rate' gives score: 8
Matching 'SP 500' with 'S&P 500 Index' gives score: 63
Matching 'SP 500' with 'Apple Stock' gives score: 12
Matching 'CPI' with 'Consumer Price Index' gives score: 26
Matching 'CPI' with 'Federal Funds Rate' gives score: 0
Matching 'CPI' with 'S&P 500 Index' gives score: 25
Matching 'CPI' with 'Apple Stock' gives score: 0
Matching 'AAPL Stock' with 'Consumer Price Index' gives 

,Indicator_Names,Matched_Names
0,CPIAUCSL,CPIAUCSL
1,FED Funds Rate,Federal Funds Rate
2,SP 500,S&P 500 Index
3,CPI,CPI
4,AAPL Stock,Apple Stock


In [32]:
# Merge Apple and Nvidia into macro dataset
combined_stock_df = apple_stock_df.merge(nvidia_stock_df, on="date", how="inner")
full_df = macro_df.reset_index().merge(combined_stock_df, on="date", how="inner").dropna()

# Set 'date' as datetime index
full_df['date'] = pd.to_datetime(full_df['date'])
full_df.set_index('date', inplace=True)

# Compute percentage changes
full_pct_df = full_df.pct_change().dropna()

# Define target returns
full_pct_df['AAPL_Returns'] = full_pct_df['AAPL']
full_pct_df['NVDA_Returns'] = full_pct_df['NVDA']
full_pct_df.drop(columns=['AAPL', 'NVDA'], inplace=True)

full_pct_df.head()




,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE,AAPL_Returns,NVDA_Returns
date,,,,,,,


In [33]:
full_pct_df

,SP500,CPIAUCSL,FEDFUNDS,A191RL1Q225SBEA,UNRATE,AAPL_Returns,NVDA_Returns
date,,,,,,,
